# 02 Data Preparation

## Melbourne Metro Rail Patronage Analytics Project

This notebook prepares the raw annual patronage files for analysis by validating, cleaning, standardising, combining, and exporting a master dataset.

## Objective

This notebook prepares the annual train patronage datasets for analysis by:

- Standardising column names
- Validating file structures
- Adding financial year information
- Combining annual datasets
- Performing data quality checks
- Creating a master dataset

In [3]:
import pandas as pd
from pathlib import Path
import re

In [4]:
PROJECT_ROOT = Path("..")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

OUTPUT_FILE = PROCESSED_DATA_DIR / "train_patronage_master.csv"

In [5]:
# Locate Raw Files
raw_files = sorted(RAW_DATA_DIR.glob("*.csv"))

raw_files

[PosixPath('../data/raw/annual_metropolitan_train_station_entries_fy_2018_2019.csv'),
 PosixPath('../data/raw/annual_metropolitan_train_station_entries_fy_2019_2020.csv'),
 PosixPath('../data/raw/annual_metropolitan_train_station_entries_fy_2020_2021.csv'),
 PosixPath('../data/raw/annual_metropolitan_train_station_entries_fy_2021_2022.csv'),
 PosixPath('../data/raw/annual_metropolitan_train_station_entries_fy_2022_2023.csv'),
 PosixPath('../data/raw/annual_metropolitan_train_station_entries_fy_2023_2024.csv'),
 PosixPath('../data/raw/annual_metropolitan_train_station_entries_fy_2024_2025.csv')]

In [6]:
# Helper Function: Extract Financial Year
def extract_financial_year(file_path):
    """
    Extract financial year from file name.
    Example:
    patronage_2018_2019.xlsx -> 2018-2019
    """
    match = re.search(r"(20\d{2})[_-](20\d{2})", file_path.stem)
    
    if match:
        return f"{match.group(1)}-{match.group(2)}"
    
    raise ValueError(f"Could not extract financial year from: {file_path.name}")

In [8]:
# Extracting Financial years from raw CSV files
for file in raw_files:
    print(file.name, "->", extract_financial_year(file))

annual_metropolitan_train_station_entries_fy_2018_2019.csv -> 2018-2019
annual_metropolitan_train_station_entries_fy_2019_2020.csv -> 2019-2020
annual_metropolitan_train_station_entries_fy_2020_2021.csv -> 2020-2021
annual_metropolitan_train_station_entries_fy_2021_2022.csv -> 2021-2022
annual_metropolitan_train_station_entries_fy_2022_2023.csv -> 2022-2023
annual_metropolitan_train_station_entries_fy_2023_2024.csv -> 2023-2024
annual_metropolitan_train_station_entries_fy_2024_2025.csv -> 2024-2025


In [9]:
# Helper Function: Standardise Column Names
def clean_column_names(df):
    """
    Standardise column names to snake_case.
    """
    df = df.copy()

    df.columns = (
        df.columns
            .str.strip()
            .str.lower()
            .str.replace(r"[^a-z0-9]+", "_", regex=True)
            .str.strip("_")
    )

    return df

In [13]:
# Load, Clean, and Combine Files
all_dataframes = []

for file in raw_files:
    financial_year = extract_financial_year(file)

    if file.suffix == ".csv":
        df_temp = pd.read_csv(file)
    else:
        raise ValueError(f"Unsupported file type: {file.suffix}")

    df_temp = clean_column_names(df_temp)
    #df_temp["financial_year"] = financial_year
    all_dataframes.append(df_temp)

master_df = pd.concat(all_dataframes, ignore_index=True)

master_df.head()

,fin_year,stop_id,stop_name,stop_lat,stop_long,pax_annual,pax_weekday,pax_norm_weekday,pax_sch_hol_weekday,pax_saturday,pax_sunday,pax_pre_am_peak,pax_am_peak,pax_interpeak,pax_pm_peak,pax_pm_late
0,FY18-19,19827,Stony Point,-38.374235,145.221837,5400,50,50,50,50,50,50,50,50,50,50
1,FY18-19,19828,Crib Point,-38.366123,145.204043,17900,50,50,50,150,100,50,50,50,50,50
2,FY18-19,19829,Morradoo,-38.354033,145.189603,5650,50,50,50,50,50,50,50,50,50,50
3,FY18-19,19830,Bittern,-38.337390,145.178026,14800,50,50,50,50,50,50,50,50,50,50
4,FY18-19,19831,Hastings,-38.305659,145.185980,21900,50,50,50,50,50,50,50,50,50,50


In [14]:
master_df.tail()

,fin_year,stop_id,stop_name,stop_lat,stop_long,pax_annual,pax_weekday,pax_norm_weekday,pax_sch_hol_weekday,pax_saturday,pax_sunday,pax_pre_am_peak,pax_am_peak,pax_interpeak,pax_pm_peak,pax_pm_late
1550,FY24-25,49458,East Pakenham,-38.084284,145.506311,54500,150,150,100,200,200,50,50,50,50,50
1551,FY24-25,52095,Southland,-37.958758,145.049124,479000,1350,1300,1500,1400,1100,50,250,450,500,100
1552,FY24-25,52159,Middle Gorge,-37.644062,145.092140,222950,750,850,500,300,200,100,350,100,250,50
1553,FY24-25,52160,Hawkstowe,-37.622992,145.097396,208500,700,750,550,350,200,150,400,100,50,50
1554,FY24-25,52161,Mernda,-37.602549,145.100883,662700,2150,2250,1800,1250,800,450,1100,400,250,50


In [16]:
# Helper function for Standardising financial year
def standardise_fin_year(value):
    """
    Standardising the finanial year column from FY24-25 -> 2024-2025
    """

    mapping = {
        "FY18-19": "2018-2019",
        "FY19-20": "2019-2020",
        "FY20-21": "2020-2021",
        "FY21-22": "2021-2022",
        "FY22-23": "2022-2023",
        "FY23-24": "2023-2024",
        "FY24-25": "2024-2025"        
    }

    return mapping.get(value, value)

In [17]:
# Standardising financial year column
master_df["financial_year"] = (
    master_df["fin_year"]
        .apply(standardise_fin_year)
)

In [18]:
master_df.head()

,fin_year,stop_id,stop_name,stop_lat,stop_long,pax_annual,pax_weekday,pax_norm_weekday,pax_sch_hol_weekday,pax_saturday,pax_sunday,pax_pre_am_peak,pax_am_peak,pax_interpeak,pax_pm_peak,pax_pm_late,financial_year
0,FY18-19,19827,Stony Point,-38.374235,145.221837,5400,50,50,50,50,50,50,50,50,50,50,2018-2019
1,FY18-19,19828,Crib Point,-38.366123,145.204043,17900,50,50,50,150,100,50,50,50,50,50,2018-2019
2,FY18-19,19829,Morradoo,-38.354033,145.189603,5650,50,50,50,50,50,50,50,50,50,50,2018-2019
3,FY18-19,19830,Bittern,-38.337390,145.178026,14800,50,50,50,50,50,50,50,50,50,50,2018-2019
4,FY18-19,19831,Hastings,-38.305659,145.185980,21900,50,50,50,50,50,50,50,50,50,50,2018-2019


In [19]:
master_df.drop(columns=["fin_year"], inplace=True)

In [20]:
# Check Master Dataset Shape
master_df.shape

(1555, 16)

In [21]:
# Validate Column Consistency
master_df.columns.tolist()

['stop_id',
 'stop_name',
 'stop_lat',
 'stop_long',
 'pax_annual',
 'pax_weekday',
 'pax_norm_weekday',
 'pax_sch_hol_weekday',
 'pax_saturday',
 'pax_sunday',
 'pax_pre_am_peak',
 'pax_am_peak',
 'pax_interpeak',
 'pax_pm_peak',
 'pax_pm_late',
 'financial_year']

In [22]:
# Investigate 2022–2023 Extra Row
row_counts = (
    master_df
    .groupby("financial_year")
    .size()
    .reset_index(name="row_count")
    .sort_values("financial_year")
)

row_counts

,financial_year,row_count
0,2018-2019,222
1,2019-2020,222
2,2020-2021,222
3,2021-2022,222
4,2022-2023,223
5,2023-2024,222
6,2024-2025,222


In [23]:
df_2022_2023 = master_df[master_df["financial_year"] == "2022-2023"]

df_2022_2023.tail(10)

,stop_id,stop_name,stop_lat,stop_long,pax_annual,pax_weekday,pax_norm_weekday,pax_sch_hol_weekday,pax_saturday,pax_sunday,pax_pre_am_peak,pax_am_peak,pax_interpeak,pax_pm_peak,pax_pm_late,financial_year
1101,19854,Flinders Street,-37.818305,144.966964,18793400,56800,56700,57200,47800,35350,1350,3300,11200,29950,10900,2022-2023
1102,19908,Richmond,-37.824074,144.990164,3105850,7850,8000,7300,11050,8750,150,850,1050,3400,2550,2022-2023
1103,19973,North Melbourne,-37.806310,144.941510,1144850,3400,3500,3100,3200,2350,150,550,750,1700,350,2022-2023
1104,19979,Jolimont,-37.816527,144.984098,921700,1950,2000,1750,3500,3750,50,150,300,800,700,2022-2023
1105,22180,Southern Cross,-37.817936,144.951411,10942400,35900,37050,31400,19550,16250,450,2400,6750,23000,4400,2022-2023
1106,19937,Moorabbin,-37.934353,145.036738,505450,1500,1550,1300,1400,1050,50,450,400,550,100,2022-2023
1107,20028,Showgrounds,-37.783500,144.915000,168400,200,50,1050,850,800,50,50,50,50,50,2022-2023
1108,20013,Bell,-37.745562,145.000155,262500,850,850,800,550,450,50,300,200,250,50,2022-2023
1109,20012,Preston,-37.738674,145.000522,398450,1250,1250,1150,950,650,50,400,350,350,50,2022-2023
1110,49343,Union,-37.823477,145.100037,50850,150,200,100,100,100,50,100,50,50,50,2022-2023


In [24]:
df_2022_2023.duplicated().sum()

np.int64(0)

In [26]:
df_2022_2023["stop_name"].duplicated().sum()

np.int64(0)

In [29]:
df_2022_2023[df_2022_2023["stop_name"].duplicated(keep=False)]

,stop_id,stop_name,stop_lat,stop_long,pax_annual,pax_weekday,pax_norm_weekday,pax_sch_hol_weekday,pax_saturday,pax_sunday,pax_pre_am_peak,pax_am_peak,pax_interpeak,pax_pm_peak,pax_pm_late,financial_year


The extra row in 2022–2023 is NOT caused by:

- Duplicate rows

- Duplicate station names

So the hypothesis is wrong.

In [30]:
df_2021_2022 = master_df[master_df["financial_year"] == "2021-2022"]

In [31]:
# Find Stations in 2022–2023 but Not 2021–2022
stations_2022 = set(df_2022_2023["stop_name"])

stations_2021 = set(df_2021_2022["stop_name"])

stations_2022 - stations_2021

{'Union'}

**The 223rd row represents a new station.**

In [32]:
master_df[
    master_df["stop_name"] == "Union"
]

,stop_id,stop_name,stop_lat,stop_long,pax_annual,pax_weekday,pax_norm_weekday,pax_sch_hol_weekday,pax_saturday,pax_sunday,pax_pre_am_peak,pax_am_peak,pax_interpeak,pax_pm_peak,pax_pm_late,financial_year
1110,49343,Union,-37.823477,145.100037,50850,150,200,100,100,100,50,100,50,50,50,2022-2023
1312,49343,Union,-37.823477,145.100037,512800,1700,1750,1450,950,650,100,1000,300,300,50,2023-2024
1549,49343,Union,-37.823477,145.100037,593100,1900,2000,1600,1100,750,100,1200,350,300,50,2024-2025


### Investigation of Row Count Difference

The 2022–2023 dataset contains 223 rows, compared to 222 rows in the surrounding years.

Analysis showed that:

- No duplicate rows exist.
- No duplicate station names exist.
- The additional row corresponds to the station "Union".

This indicates the row count difference is due to a network change rather than a data quality issue.

But after 2023 the number of rows count back again to 222. Why?

In [33]:
# Data Type Review
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1555 entries, 0 to 1554
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   stop_id              1555 non-null   int64  
 1   stop_name            1555 non-null   str    
 2   stop_lat             1555 non-null   float64
 3   stop_long            1555 non-null   float64
 4   pax_annual           1555 non-null   int64  
 5   pax_weekday          1555 non-null   int64  
 6   pax_norm_weekday     1555 non-null   int64  
 7   pax_sch_hol_weekday  1555 non-null   int64  
 8   pax_saturday         1555 non-null   int64  
 9   pax_sunday           1555 non-null   int64  
 10  pax_pre_am_peak      1555 non-null   int64  
 11  pax_am_peak          1555 non-null   int64  
 12  pax_interpeak        1555 non-null   int64  
 13  pax_pm_peak          1555 non-null   int64  
 14  pax_pm_late          1555 non-null   int64  
 15  financial_year       1555 non-null   str    
dtyp

In [34]:
master_df[
    master_df["stop_name"].isin(
        ["Union", "Mont Albert", "Surrey Hills"]
    )
][
    ["financial_year", "stop_name"]
].sort_values(
    ["stop_name", "financial_year"]
)

,financial_year,stop_name
65,2018-2019,Mont Albert
287,2019-2020,Mont Albert
508,2020-2021,Mont Albert
731,2021-2022,Mont Albert
953,2022-2023,Mont Albert
64,2018-2019,Surrey Hills
286,2019-2020,Surrey Hills
507,2020-2021,Surrey Hills
730,2021-2022,Surrey Hills
952,2022-2023,Surrey Hills


**Further investigation showed:**

- Mont Albert and Surrey Hills are present from 2018–2019 to 2022–2023.
- Union first appears in 2022–2023.
- Mont Albert and Surrey Hills are no longer present from 2023–2024 onwards.

This indicates that 2022–2023 represents a transition period in the rail network where Union Station was introduced before the removal of Mont Albert and Surrey Hills. Therefore, the additional row reflects a genuine network change rather than a data quality issue.

In [35]:
# Master data missing value check
missing_values = (
    master_df
    .isna()
    .sum()
    .reset_index()
)

missing_values.columns = ["column", "missing_count"]

missing_values.sort_values("missing_count", ascending=False)

,column,missing_count
0,stop_id,0
1,stop_name,0
2,stop_lat,0
3,stop_long,0
4,pax_annual,0
5,pax_weekday,0
6,pax_norm_weekday,0
7,pax_sch_hol_weekday,0
8,pax_saturday,0
9,pax_sunday,0


In [36]:
# Duplicate Check
master_df.duplicated().sum()

np.int64(0)

In [37]:
# Basic Numeric Validation
numeric_columns = master_df.select_dtypes(include="number").columns.tolist()

numeric_columns

['stop_id',
 'stop_lat',
 'stop_long',
 'pax_annual',
 'pax_weekday',
 'pax_norm_weekday',
 'pax_sch_hol_weekday',
 'pax_saturday',
 'pax_sunday',
 'pax_pre_am_peak',
 'pax_am_peak',
 'pax_interpeak',
 'pax_pm_peak',
 'pax_pm_late']

In [38]:
master_df[numeric_columns].describe()

,stop_id,stop_lat,stop_long,pax_annual,pax_weekday,pax_norm_weekday,pax_sch_hol_weekday,pax_saturday,pax_sunday,pax_pre_am_peak,pax_am_peak,pax_interpeak,pax_pm_peak,pax_pm_late
count,1555.000000,1555.000000,1555.000000,1.555000e+03,1555.000000,1555.00000,1555.000000,1555.000000,1555.000000,1555.000000,1555.000000,1555.000000,1555.000000,1555.000000
mean,21339.065595,-37.852077,145.045012,6.973038e+05,2269.067524,2350.96463,1967.363344,1315.787781,956.688103,150.610932,650.225080,491.800643,869.549839,214.501608
std,6199.167129,0.139769,0.140502,1.786096e+06,5850.933120,6011.70986,5266.377360,3403.431052,2539.683176,164.050090,620.850406,1150.186868,3571.496329,938.381671
min,15351.000000,-38.374235,144.661118,1.500000e+03,50.000000,50.00000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000
25%,19880.000000,-37.899378,144.960545,1.882000e+05,600.000000,650.00000,500.000000,350.000000,250.000000,50.000000,250.000000,150.000000,100.000000,50.000000
50%,19936.000000,-37.825894,145.036439,3.492000e+05,1100.000000,1150.00000,950.000000,650.000000,500.000000,100.000000,500.000000,250.000000,250.000000,50.000000
75%,19995.000000,-37.769865,145.121429,6.128250e+05,2000.000000,2050.00000,1700.000000,1200.000000,850.000000,200.000000,850.000000,450.000000,450.000000,100.000000
max,52161.000000,-37.579091,145.506311,2.832065e+07,93050.000000,94900.00000,85900.000000,52050.000000,38750.000000,1900.000000,5400.000000,16650.000000,55250.000000,16250.000000


In [39]:
negative_value_check = {}

for column in numeric_columns:
    negative_value_check[column] = (master_df[column] < 0).sum()

pd.DataFrame(
    negative_value_check.items(),
    columns=["column", "negative_value_count"]
)

,column,negative_value_count
0,stop_id,0
1,stop_lat,1555
2,stop_long,0
3,pax_annual,0
4,pax_weekday,0
5,pax_norm_weekday,0
6,pax_sch_hol_weekday,0
7,pax_saturday,0
8,pax_sunday,0
9,pax_pre_am_peak,0


Latitude has negative values which is normal for Melbourne region

In [40]:
# Changing final column order

front_columns = ["financial_year"]

remaining_columns = [
    col for col in master_df.columns
    if col not in front_columns
]

master_df = master_df[front_columns + remaining_columns]

master_df.head()

,financial_year,stop_id,stop_name,stop_lat,stop_long,pax_annual,pax_weekday,pax_norm_weekday,pax_sch_hol_weekday,pax_saturday,pax_sunday,pax_pre_am_peak,pax_am_peak,pax_interpeak,pax_pm_peak,pax_pm_late
0,2018-2019,19827,Stony Point,-38.374235,145.221837,5400,50,50,50,50,50,50,50,50,50,50
1,2018-2019,19828,Crib Point,-38.366123,145.204043,17900,50,50,50,150,100,50,50,50,50,50
2,2018-2019,19829,Morradoo,-38.354033,145.189603,5650,50,50,50,50,50,50,50,50,50,50
3,2018-2019,19830,Bittern,-38.337390,145.178026,14800,50,50,50,50,50,50,50,50,50,50
4,2018-2019,19831,Hastings,-38.305659,145.185980,21900,50,50,50,50,50,50,50,50,50,50


In [41]:
# Save Processed Master Dataset
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

master_df.to_csv(OUTPUT_FILE, index=False)

print(f"Master dataset saved to: {OUTPUT_FILE}")

Master dataset saved to: ../data/processed/train_patronage_master.csv


In [42]:
# Final Validation After Export
validation_df = pd.read_csv(OUTPUT_FILE)

validation_df.shape

(1555, 16)

In [43]:
validation_df.head()

,financial_year,stop_id,stop_name,stop_lat,stop_long,pax_annual,pax_weekday,pax_norm_weekday,pax_sch_hol_weekday,pax_saturday,pax_sunday,pax_pre_am_peak,pax_am_peak,pax_interpeak,pax_pm_peak,pax_pm_late
0,2018-2019,19827,Stony Point,-38.374235,145.221837,5400,50,50,50,50,50,50,50,50,50,50
1,2018-2019,19828,Crib Point,-38.366123,145.204043,17900,50,50,50,150,100,50,50,50,50,50
2,2018-2019,19829,Morradoo,-38.354033,145.189603,5650,50,50,50,50,50,50,50,50,50,50
3,2018-2019,19830,Bittern,-38.337390,145.178026,14800,50,50,50,50,50,50,50,50,50,50
4,2018-2019,19831,Hastings,-38.305659,145.185980,21900,50,50,50,50,50,50,50,50,50,50


## Data Preparation Summary

The Melbourne metropolitan train station patronage datasets for financial years 2018–2019 to 2024–2025 were successfully prepared for analysis.

Key preparation activities included:

- Loading and validating all seven annual patronage datasets.
- Standardising column names to improve consistency and readability.
- Reviewing data types, missing values, and duplicate records.
- Combining all annual datasets into a single master dataset.
- Creating a consistent financial year field to support trend analysis.
- Performing data quality checks on row counts and station records.
- Exporting the cleaned master dataset to the processed data folder for downstream analysis.

### Key Findings During Data Preparation

- All datasets contained a consistent structure with 16 columns.
- No duplicate records were identified.
- The 2022–2023 dataset contained 223 records, while all other years contained 222 records.
- Investigation confirmed that the additional record was not a data quality issue.
- The additional record corresponded to Union Station, which first appeared in the 2022–2023 dataset.
- Mont Albert and Surrey Hills stations remained present during 2022–2023 and were removed from subsequent years, indicating a network transition period.
- The row count variation therefore reflects a genuine change in the metropolitan rail network rather than a data quality anomaly.

The resulting master dataset is considered suitable for exploratory analysis and business reporting.

## Next Steps

With the master dataset prepared and validated, the next phase of the project will focus on exploratory data analysis (EDA) to understand passenger demand patterns across Melbourne's metropolitan rail network.

The analysis will investigate:

- Overall network patronage trends between 2018–2019 and 2024–2025.
- The impact of COVID-19 on train station usage.
- Recovery patterns following the pandemic period.
- The highest and lowest patronage stations across the network.
- Station-level growth and decline over time.
- Changes in weekday, weekend, and peak-period passenger behaviour.
- Emerging demand trends that may inform future transport planning and operational decision-making.

Insights generated during this phase will form the foundation for SQL analysis, Power BI dashboard development, and final business recommendations.